# Sesión 6 mejorada — Web Scraping con Python

**Curso:** Fundamentos de Programación para IA Generativa Aplicada  
**Programa:** Especialización en IA Generativa aplicada a CCSS y Gestión Pública  

## Propósito de la sesión

Esta versión mejorada de la sesión 6 reproduce y organiza el notebook original de web scraping, con una orientación más rigurosa, reproducible y útil para proyectos aplicados.
La sesión se estructura en dos niveles:

1. **Scraping estático**, usando `requests` y `BeautifulSoup`.
2. **Scraping dinámico**, usando `Selenium`.

## Idea metodológica

El web scraping no debe entenderse solo como “extraer datos”, sino como una secuencia ordenada de decisiones:

**sitio web → inspección estructural → elección de herramienta → prueba → extracción → almacenamiento → auditoría**

En esta sesión no desarrollaremos todavía la Tarea 2, pero sí construiremos la base técnica y metodológica que luego permitirá adaptarla al portal OECE.

## Ruta de trabajo

Todos los resultados de esta sesión se guardarán en:

`Mi unidad -> IA Generativa para CCSS -> Base de datos -> Resultados -> Sesion_6_Webscraping_mejorada`

In [1]:
# ============================================================
# SESIÓN 6 MEJORADA — CONFIGURACIÓN DEL ENTORNO
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import json
import time
import shutil

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 200)

RUTA_BASE = Path("/content/drive/MyDrive/IA Generativa para CCSS/Base de datos")
RUTA_INSUMOS = RUTA_BASE / "Insumos"
RUTA_RESULTADOS = RUTA_BASE / "Resultados" / "Sesion_6_Webscraping_mejorada"

RUTA_TABLAS = RUTA_RESULTADOS / "tablas"
RUTA_HTML = RUTA_RESULTADOS / "html"
RUTA_IMAGENES = RUTA_RESULTADOS / "imagenes"
RUTA_PDFS = RUTA_RESULTADOS / "pdfs"
RUTA_AUDITORIA = RUTA_RESULTADOS / "auditoria"

for ruta in [RUTA_RESULTADOS, RUTA_TABLAS, RUTA_HTML, RUTA_IMAGENES, RUTA_PDFS, RUTA_AUDITORIA]:
    ruta.mkdir(parents=True, exist_ok=True)

print("RUTA_BASE      :", RUTA_BASE)
print("RUTA_INSUMOS   :", RUTA_INSUMOS)
print("RUTA_RESULTADOS:", RUTA_RESULTADOS)

Mounted at /content/drive
RUTA_BASE      : /content/drive/MyDrive/IA Generativa para CCSS/Base de datos
RUTA_INSUMOS   : /content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Insumos
RUTA_RESULTADOS: /content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada


## Marco metodológico de la sesión

La sesión original define el scraping como un proceso de extracción de datos desde páginas web y propone una secuencia típica: identificación del sitio, inspección de estructura, selección de herramienta, escritura de código, testing, extracción y almacenamiento. Esa secuencia será respetada aquí, pero de forma más explícita y exportable. :contentReference[oaicite:6]{index=6}

También se mantiene la distinción central entre:
- **páginas estáticas**, que entregan el contenido en el HTML inicial;
- **páginas dinámicas**, que requieren ejecución de JavaScript y, por tanto, navegación automatizada con Selenium.

In [2]:
# ============================================================
# INSTALACIÓN DE PAQUETES
# ============================================================

!pip -q install requests beautifulsoup4 pandas selenium webdriver-manager lxml openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 7.7 MB/s eta 0:00:00


In [3]:
# ============================================================
# SCRAPING ESTÁTICO — WIKIPEDIA
# ============================================================

import requests
from bs4 import BeautifulSoup

url = "https://es.wikipedia.org/wiki/Anexo:Pa%C3%ADses_por_tasa_de_suicidio"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

html_text = response.text

with open(RUTA_HTML / "wikipedia_suicidio.html", "w", encoding="utf-8") as f:
    f.write(html_text)

soup = BeautifulSoup(html_text, "html.parser")

print("Status code:", response.status_code)
print("Título de página:", soup.title.text.strip() if soup.title else "Sin título")

Status code: 200
Título de página: Anexo:Países por tasa de suicidio - Wikipedia, la enciclopedia libre


In [4]:
# ============================================================
# EXTRACCIÓN DE TABLAS
# ============================================================

tables = soup.find_all("table")
print("Número total de tablas encontradas:", len(tables))

table = tables[2]
rows = table.find_all("tr")

headers_table = [th.text.strip() for th in rows[0].find_all("th")]

data = []
for row in rows[1:]:
    cols = row.find_all("td")
    row_data = [col.text.strip() for col in cols]
    if row_data:
        data.append(row_data)

df_wiki = pd.DataFrame(data, columns=headers_table)
display(df_wiki.head())

df_wiki.to_csv(RUTA_TABLAS / "s6_wikipedia_tabla_principal.csv", index=False, encoding="utf-8")

Número total de tablas encontradas: 4


,Regiones,Masculino,Femenino,Promedio,Año
0,Groenlandia,"84,99","28,44","58,28 (0,0583%)",2016
1,Lituania,"65,1","12,4","28,27 (0,02827%)",2016
2,Corea del Sur,"38,5","14,8","26,6 (0,0266%)",2018
3,Guyana,"41,25","10,20","25,52 (0,0255%)",2017
4,Kazajistán,"40,68","8,01","23,81 (0,0238%)",2017


In [5]:
# ============================================================
# METADATOS Y ENLACES
# ============================================================

page_title = soup.title.text.strip() if soup.title else None
canonical_tag = soup.find("link", {"rel": "canonical"})
canonical_url = canonical_tag["href"] if canonical_tag else url
links = [a["href"] for a in soup.find_all("a", href=True)]

tabla_meta = pd.DataFrame([
    {"campo": "page_title", "valor": page_title},
    {"campo": "canonical_url", "valor": canonical_url},
    {"campo": "n_links", "valor": len(links)},
    {"campo": "example_link", "valor": links[0] if links else None},
])

display(tabla_meta)
tabla_meta.to_csv(RUTA_AUDITORIA / "s6_wikipedia_metadatos.csv", index=False, encoding="utf-8")

df_links = pd.DataFrame({"href": links})
df_links.to_csv(RUTA_AUDITORIA / "s6_wikipedia_links.csv", index=False, encoding="utf-8")

,campo,valor
0,page_title,"Anexo:Países por tasa de suicidio - Wikipedia, la enciclopedia libre"
1,canonical_url,https://es.wikipedia.org/wiki/Anexo:Pa%C3%ADses_por_tasa_de_suicidio
2,n_links,769
3,example_link,#bodyContent


In [6]:
# ============================================================
# DESCARGA DE IMÁGENES
# ============================================================

flag_images = soup.select("span.flagicon img")
print("Imágenes detectadas:", len(flag_images))

descargadas = []

for i, img in enumerate(flag_images[:20], start=1):
    img_url = "https:" + img["src"]
    img_name = img_url.split("/")[-1]
    img_path = RUTA_IMAGENES / img_name

    img_data = requests.get(img_url, headers=headers, timeout=30).content
    with open(img_path, "wb") as handler:
        handler.write(img_data)

    descargadas.append({"n": i, "img_url": img_url, "archivo": str(img_path)})

df_imgs = pd.DataFrame(descargadas)
display(df_imgs.head())
df_imgs.to_csv(RUTA_AUDITORIA / "s6_imagenes_descargadas.csv", index=False, encoding="utf-8")

Imágenes detectadas: 183


,n,img_url,archivo
0,1,https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Flag_of_Lesotho.svg/20px-Flag_of_Lesotho.svg.png,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/imagenes/20px-Flag_of_Lesotho.svg.png
1,2,https://upload.wikimedia.org/wikipedia/commons/thumb/9/99/Flag_of_Guyana.svg/20px-Flag_of_Guyana.svg.png,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/imagenes/20px-Flag_of_Guyana.svg.png
2,3,https://upload.wikimedia.org/wikipedia/commons/thumb/f/fb/Flag_of_Eswatini.svg/20px-Flag_of_Eswatini.svg.png,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/imagenes/20px-Flag_of_Eswatini.svg.png
3,4,https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/Flag_of_Kiribati.svg/20px-Flag_of_Kiribati.svg.png,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/imagenes/20px-Flag_of_Kiribati.svg.png
4,5,https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Flag_of_the_Federated_States_of_Micronesia.svg/20px-Flag_of_the_Federated_States_of_Micronesia.svg.png,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/imagenes/20px-Flag_of_the_Federated_States_of_Micronesia.svg.png


In [7]:
# ============================================================
# SELENIUM — CONFIGURACIÓN
# ============================================================

!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y
!pip -q install selenium webdriver-manager pandas

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

opciones = Options()
opciones.add_argument("--headless")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")
opciones.add_argument("--window-size=1920,1080")
opciones.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

chrome_path = shutil.which("google-chrome") or shutil.which("google-chrome-stable")
if chrome_path:
    opciones.binary_location = chrome_path

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opciones)
wait = WebDriverWait(driver, 20)

print("Driver inicializado correctamente.")

Selecting previously unselected package google-chrome-stable.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (147.0.7727.55-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-stable depends on libvulkan1; however:
  Package libvulkan1 is not installed.
 google-chrome-stable depends on libxcomposite1 (>= 1:0.4.4-1); however:
  Package libxcomposite1 is not installed.

dpkg: error processing package google-chrome-stable (--install):
 dependency problems - leaving unconfigured
Processing trigger

In [8]:
# ============================================================
# CASO DINÁMICO — EL PERUANO
# ============================================================

url = "https://diariooficial.elperuano.pe/normas"
driver.get(url)

wait.until(EC.presence_of_element_located((By.CLASS_NAME, "edicionesoficiales_articulos")))
static_articles = driver.find_elements(By.CSS_SELECTOR, "article.edicionesoficiales_articulos")

static_results = []
for article in static_articles:
    lines = article.text.split("\n")
    if len(lines) >= 2:
        static_results.append({
            "sector": lines[0],
            "title": lines[1],
            "content": " ".join(lines[2:])
        })

df_static = pd.DataFrame(static_results)
display(df_static.head())
df_static.to_excel(RUTA_TABLAS / "s6_elperuano_static.xlsx", index=False)

,sector,title,content
0,PRESIDENCIA DEL CONSEJO DE MINISTROS,RESOLUCION JEFATURAL N° 000059-2026-ANIN/JEF,Fecha: 07/04/2026 Aprueban ejecución de la expropiación de área afectada de inmueble para el Proyecto: Creación del servicio de protección en las riberas del Río Huaura vulnerable ante peligro de ...
1,DESARROLLO AGRARIO Y RIEGO,RESOLUCION DIRECTORAL N° 066-2026-MIDAGRI-DVDAFIR-AGRO RURAL-DE,Fecha: 07/04/2026 Designan Jefe de la Subunidad de Infraestructura de Riego Menor de la Unidad de Infraestructura Rural del Programa de Desarrollo Productivo Agrario Rural – AGRO RURAL
2,DESARROLLO E INCLUSIÓN SOCIAL,RESOLUCION SUPREMA N° 008-2026-MIDIS,Fecha: 07/04/2026 Aceptan renuncia de Director Ejecutivo del Programa Nacional de Asistencia Solidaria Pensión 65
3,DESARROLLO E INCLUSIÓN SOCIAL,RESOLUCION SUPREMA N° 009-2026-MIDIS,Fecha: 07/04/2026 Designan Directora Ejecutiva del Programa Nacional de Asistencia Solidaria Pensión 65
4,ECONOMIA Y FINANZAS,RESOLUCION MINISTERIAL N° 167-2026-EF/49,Fecha: 07/04/2026 Designan Asesor de Secretaría General II del Ministerio


In [9]:
# ============================================================
# BÚSQUEDA DINÁMICA POR FECHA
# ============================================================

target_date = "20/01/2026"

input_from = driver.find_element(By.ID, "cddesde")
input_to = driver.find_element(By.ID, "cdhasta")
search_btn = driver.find_element(By.ID, "btnBuscar")

driver.execute_script(f"arguments[0].value = '{target_date}';", input_from)
driver.execute_script(f"arguments[0].value = '{target_date}';", input_to)
driver.execute_script("arguments[0].click();", search_btn)

time.sleep(7)

dynamic_articles = driver.find_elements(By.CSS_SELECTOR, "article.edicionesoficiales_articulos")

dynamic_results = []
for article in dynamic_articles:
    lines = article.text.split("\n")
    if len(lines) >= 2:
        dynamic_results.append({
            "date_searched": target_date,
            "sector": lines[0],
            "title": lines[1],
            "content": " ".join(lines[2:])
        })

df_dynamic = pd.DataFrame(dynamic_results)
display(df_dynamic.head())
df_dynamic.to_excel(RUTA_TABLAS / f"s6_elperuano_dynamic_{target_date.replace('/','-')}.xlsx", index=False)

,date_searched,sector,title,content
0,20/01/2026,AMBIENTE,RESOLUCION N° 006-2026-SENAMHI/PREJ,Fecha: 20/01/2026 Edición Extraordinaria Designan asesora de Alta Dirección para la Presidencia Ejecutiva del Servicio Nacional de Meteorología e Hidrología del Perú – SENAMHI
1,20/01/2026,CULTURA,RESOLUCION MINISTERIAL N° 000014-2026-MC,Fecha: 20/01/2026 Edición Extraordinaria Designan Director de Programa Sectorial III de la Dirección de Patrimonio Histórico Inmueble de la Dirección General de Patrimonio Cultural del Ministerio ...
2,20/01/2026,DESARROLLO E INCLUSIÓN SOCIAL,RESOLUCION MINISTERIAL N° D000009-2026-MIDIS,Fecha: 20/01/2026 Edición Extraordinaria Aprueban Lineamientos N° 001-2026-MIDIS denominados Lineamientos para la atención alimentaria y entrega de subsidios económicos del Programa de Complementa...
3,20/01/2026,INTERIOR,RESOLUCION MINISTERIAL N° 0064-2026-IN,Fecha: 20/01/2026 Edición Extraordinaria Autorizan viaje de personal policial a la República Argentina en comisión de servicios
4,20/01/2026,INTERIOR,RESOLUCION MINISTERIAL N° 0065-2026-IN,Fecha: 20/01/2026 Edición Extraordinaria Autorizan viaje de personal policial a la República Argentina en comisión de servicios


In [10]:
# ============================================================
# DESCARGA DE PDFs DESDE __NEXT_DATA__
# ============================================================

base_url = "https://diariooficial.elperuano.pe"
request_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}

detail_urls = []
for article_element in dynamic_articles[:10]:
    try:
        detail_link = article_element.find_element(By.TAG_NAME, "a").get_attribute("href")
        detail_urls.append(detail_link)
    except:
        continue

pdf_descargas = []

for detail_url in detail_urls:
    driver.get(detail_url)
    try:
        script_tag = wait.until(EC.presence_of_element_located((By.ID, "__NEXT_DATA__")))
        json_data = json.loads(script_tag.get_attribute("innerHTML"))
        full_pdf_url = json_data["props"]["pageProps"]["dispositivo"]["urlPDF"]

        pdf_filename_base = full_pdf_url.split("/")[-2]
        pdf_filename = f"{pdf_filename_base}.pdf"
        pdf_path = RUTA_PDFS / pdf_filename

        pdf_response = requests.get(full_pdf_url, headers=request_headers, timeout=30)
        pdf_response.raise_for_status()

        with open(pdf_path, "wb") as f:
            f.write(pdf_response.content)

        pdf_descargas.append({
            "detail_url": detail_url,
            "pdf_url": full_pdf_url,
            "archivo": str(pdf_path)
        })

        time.sleep(1)

    except Exception as e:
        pdf_descargas.append({
            "detail_url": detail_url,
            "pdf_url": None,
            "archivo": None,
            "error": str(e)
        })

df_pdfs = pd.DataFrame(pdf_descargas)
display(df_pdfs.head())
df_pdfs.to_csv(RUTA_AUDITORIA / "s6_pdfs_descargados.csv", index=False, encoding="utf-8")

,detail_url,pdf_url,archivo
0,https://busquedas.elperuano.pe/dispositivo/NL/2478860-1,https://busquedas.elperuano.pe/api/media/http://172.20.0.101/file/2oWY3JhuarvAwRcILESB3O/*/2478860-1.PDF/PDF,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/pdfs/2478860-1.PDF.pdf
1,https://busquedas.elperuano.pe/dispositivo/NL/2478740-1,https://busquedas.elperuano.pe/api/media/http://172.20.0.101/file/45SBedk9qpKAuwlfVvdjQM/*/2478740-1.PDF/PDF,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/pdfs/2478740-1.PDF.pdf
2,https://busquedas.elperuano.pe/dispositivo/NL/2478990-1,https://busquedas.elperuano.pe/api/media/http://172.20.0.101/file/5KKdRgTK4A693OBkJ7REQh/*/2478990-1.PDF/PDF,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/pdfs/2478990-1.PDF.pdf
3,https://busquedas.elperuano.pe/dispositivo/NL/2478982-1,https://busquedas.elperuano.pe/api/media/http://172.20.0.101/file/95S0OU264QyAh8hJZ1H-nP/*/2478982-1.PDF/PDF,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/pdfs/2478982-1.PDF.pdf
4,https://busquedas.elperuano.pe/dispositivo/NL/2478989-1,https://busquedas.elperuano.pe/api/media/http://172.20.0.101/file/92q2MpfYK8qAdFtnU-sd1Z/*/2478989-1.PDF/PDF,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/pdfs/2478989-1.PDF.pdf


In [11]:
# ============================================================
# CIERRE DEL NAVEGADOR
# ============================================================

driver.quit()
print("Driver cerrado correctamente.")

Driver cerrado correctamente.


In [12]:
# ============================================================
# RESUMEN FINAL
# ============================================================

resumen = pd.DataFrame({
    "objeto": [
        "tabla wikipedia",
        "metadatos wikipedia",
        "links wikipedia",
        "imagenes wikipedia",
        "resultados estaticos elperuano",
        "resultados dinamicos elperuano",
        "pdfs descargados"
    ],
    "ruta": [
        str(RUTA_TABLAS / "s6_wikipedia_tabla_principal.csv"),
        str(RUTA_AUDITORIA / "s6_wikipedia_metadatos.csv"),
        str(RUTA_AUDITORIA / "s6_wikipedia_links.csv"),
        str(RUTA_AUDITORIA / "s6_imagenes_descargadas.csv"),
        str(RUTA_TABLAS / "s6_elperuano_static.xlsx"),
        str(RUTA_TABLAS / f"s6_elperuano_dynamic_20-01-2026.xlsx"),
        str(RUTA_AUDITORIA / "s6_pdfs_descargados.csv"),
    ]
})

display(resumen)
resumen.to_csv(RUTA_RESULTADOS / "s6_resumen_exportaciones.csv", index=False, encoding="utf-8")

,objeto,ruta
0,tabla wikipedia,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/tablas/s6_wikipedia_tabla_principal.csv
1,metadatos wikipedia,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/auditoria/s6_wikipedia_metadatos.csv
2,links wikipedia,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/auditoria/s6_wikipedia_links.csv
3,imagenes wikipedia,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/auditoria/s6_imagenes_descargadas.csv
4,resultados estaticos elperuano,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/tablas/s6_elperuano_static.xlsx
5,resultados dinamicos elperuano,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/tablas/s6_elperuano_dynamic_20-01-2026.xlsx
6,pdfs descargados,/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_6_Webscraping_mejorada/auditoria/s6_pdfs_descargados.csv


## Cierre metodológico de la sesión 6

La sesión 6 mostró que el web scraping exige distinguir entre dos tipos de arquitectura web:

- páginas estáticas, donde el HTML inicial contiene la información suficiente;
- páginas dinámicas, donde el contenido depende de JavaScript y requiere automatización del navegador.

En esta versión mejorada, la sesión fue reorganizada como flujo reproducible: configuración, inspección, extracción, descarga y exportación. Esto la vuelve más útil no solo como demostración técnica, sino como preparación metodológica para proyectos aplicados de captura de datos administrativos. La principal lección es que el scraping no comienza con el código, sino con la inspección estructural del sitio y la decisión correcta sobre la herramienta a emplear.